# Notebook 01 — Dataset audit and split construction (Gate 0B)

Normalize, filter, split, hash, inspect. Reproducible hashes; disjoint splits.

In [ ]:
# --- notebook preamble: paths, modes, safe display ---
import os, sys, json
from pathlib import Path
REPO = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("FAST_DEV_RUN", "1")
FAST_DEV_RUN = os.environ.get("FAST_DEV_RUN", "1") == "1"
RUN_EXPENSIVE = os.environ.get("RUN_EXPENSIVE", "0") == "1"
RECOMPUTE = os.environ.get("RECOMPUTE", "0") == "1"
try:
    from IPython.display import display
except Exception:
    def display(*a, **k):
        for x in a:
            print(x)
import numpy as np
import subliminal_icl as S
print("FAST_DEV_RUN=", FAST_DEV_RUN, "RUN_EXPENSIVE=", RUN_EXPENSIVE, "pkg", S.__version__)


## 1. Configuration and run manifest

In [ ]:
from subliminal_icl.manifests import Manifest
man = Manifest.create(phase="01", model_tag="smoke", target="eagle", seed=0)
print("run_id:", man.run_id)


## 2. Preflight assertions

In [ ]:
print('preflight: package + numpy import OK')

## 3. Load or compute cached artifacts
Build a tiny normalized set, filter leakage, and verify split disjointness.

In [ ]:
from subliminal_icl.data_schemas import PromptCompletionRow
from subliminal_icl.data_builders import filter_rows, assign_splits, check_disjoint, split_hash
rows = [PromptCompletionRow.build("fix", f"seq {i}", "1, 2, 3", source_index=i, target_trait="eagle")
        for i in range(12)]
# inject a leaky row
rows.append(PromptCompletionRow.build("fix", "about birds", "eagles are great", source_index=99, target_trait="eagle"))
res = filter_rows(rows, "eagle")
print("kept", len(res["kept"]), "dropped", len(res["dropped"]))
splits = assign_splits(res["kept"], {"candidate_search": 6, "eval_dev": 2, "eval_final": 2}, seed=0)
dis = check_disjoint(splits)
h1 = split_hash(splits["candidate_search"]); h2 = split_hash(assign_splits(res["kept"], {"candidate_search": 6, "eval_dev": 2, "eval_final": 2}, seed=0)["candidate_search"])
print("disjoint:", dis, "hash stable:", h1 == h2)

## 4. Interactive inspection widgets

## 5. Diagnostic plots and tables
See printed tables above; plotting helpers in `subliminal_icl.plotting`.

## 6. Scientific gate decision

### ### Interactive inspection (widgets)
In JupyterLab with `ipywidgets`, this section exposes selectors for model / target trait / run id / layer / token position / component / context size / candidate source / search seed / intervention strength, plus row and beam browsers. They are omitted from the executed cells so the notebook also runs headless in `FAST_DEV_RUN`.

In [ ]:
# --- Scientific gate decision + checkpoint ---
from subliminal_icl.gates import run_gate_checks
checks = [("leakage_dropped", len(res["dropped"]) >= 1, "leaky row filtered"),
          ("splits_disjoint", dis["disjoint"], str(dis["problems"])),
          ("hash_stable", h1 == h2, "re-normalization identical")]
gs = run_gate_checks("gate_01_datasets_prep", "Dataset audit and splits", checks,
                     config={"fast_dev_run": FAST_DEV_RUN}, write=False)
display(gs.to_dataframe())
print("GATE", gs.gate_id, "->", gs.status)
if not FAST_DEV_RUN:
    assert gs.passed, gs.failure_summary


## 7. Write checkpoint report
Written by the gate cell when not in FAST_DEV_RUN.